In [2]:
import sys
sys.path.append('..')

import pandas as pd
from src.module3_llm.llm_client import RetailLLMClient
from src.module3_llm.churn_narrator import ChurnNarrator
from src.module3_llm.demand_narrator import DemandNarrator
from src.module3_llm.query_engine import QueryEngine
from src.module3_llm.winback_generator import WinBackGenerator
from src.module3_llm.prompt_templates import (
    CHURN_EXPLANATION_TEMPLATE,
    DEMAND_FORECAST_SUMMARY_TEMPLATE,
    NL_QUERY_ROUTER_TEMPLATE
)

client = RetailLLMClient()
narrator = ChurnNarrator(client)

Initializing LLM client: llama3
LLM client ready


In [3]:
# Test 1: Churn explanation
import pandas as pd

print("="*50)
print("TEST 1: Churn explanation")

# Build a sample customer + shap row manually to match the current data schema
sample_customer = pd.Series({
    'Frequency': 3,
    'Monetary': 145.50,
    'AvgOrderValue': 48.50,
    'UniqueProducts': 12,
    'DaysActive': 45,
    'ChurnProbability': 0.68
})

sample_shap = pd.Series({
    'shap_Frequency': 0.15,
    'shap_Monetary': 0.05,
    'shap_AvgOrderValue': 0.20,
    'shap_UniqueProducts': -0.02,
    'shap_DaysActive': 0.30,
    'shap_OrdersPerDay': 0.01
})

feature_cols = ['Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'DaysActive', 'OrdersPerDay']

response = narrator.explain_customer(sample_customer, sample_shap, feature_cols)
print(f"Response: {response}")

TEST 1: Churn explanation
Response: This customer's orders are spread out over a longer period, which the model associates with elevated churn risk, even though they remain an active, repeat customer with 3.0 orders. The wide purchase window between their first and last order is a significant concern, as it suggests that this customer may be experiencing difficulties or distractions that could lead to them abandoning their relationship with the store.


In [5]:
# Test 2: Demand summary
print("="*50)
print("TEST 2: Demand forecast summary")

reliability_instruction = (
    "This forecast is considered reliable. You may state the forecasted "
    "demand number normally."
)

response2 = client.generate_from_template(
    DEMAND_FORECAST_SUMMARY_TEMPLATE,
    {
        'product_name': 'White Hanging Heart T-Light Holder',
        'stock_code': '85123A',
        'current_stock': 45,
        'forecast_weeks': 4,
        'forecasted_demand': 120,
        'lower_bound': 85,
        'upper_bound': 155,
        'alert_status': 'Stockout Risk',
        'trend_direction': 'Growing',
        'reliability_instruction': reliability_instruction
    },
    max_words=60
)
print(f"Response: {response2}")

TEST 2: Demand forecast summary
Response: The White Hanging Heart T-Light Holder is currently at risk of selling out due to a significant increase in demand over the next four weeks, with forecasted sales reaching 120 units. To avoid stockouts and ensure customer satisfaction, we need to replenish our inventory levels to meet this growing demand, ideally before it's too late.


In [6]:
# Test 3: Query routing
print("\n" + "="*50)
print("TEST 3: Query routing")
questions = [
    "Which customers are about to leave?",
    "What products need reordering?",
    "Show me revenue at risk and low stock items",
    "What is the weather today?"
]

for q in questions:
    route = client.generate_from_template(
        NL_QUERY_ROUTER_TEMPLATE,
        {'question': q},
        max_words=5
    )
    print(f"Q: {q}")
    print(f"-> Route: {route.strip()}\n")


TEST 3: Query routing
Q: Which customers are about to leave?
-> Route: CHURN

Q: What products need reordering?
-> Route: DEMAND

Q: Show me revenue at risk and low stock items
-> Route: BOTH

Q: What is the weather today?
-> Route: UNKNOWN



In [7]:
import pandas as pd
from src.module3_llm.llm_client import RetailLLMClient
from src.module3_llm.churn_narrator import ChurnNarrator

client = RetailLLMClient()
narrator = ChurnNarrator(client)

risk_table = pd.read_csv('../data/processed/customer_risk_table.csv')
shap_values_df = pd.read_csv('../data/processed/shap_values.csv')

feature_cols = [
    'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts',
    'AvgQuantity', 'DaysActive', 'OrdersPerDay'
]

high_risk_customers = risk_table[risk_table['RiskTier'] == '🔴 High Risk']
sample_idx = high_risk_customers.index[0]
customer_row = risk_table.loc[sample_idx]
shap_row = shap_values_df.iloc[sample_idx] if sample_idx < len(shap_values_df) else pd.Series()

winback_email = narrator.generate_winback(
    customer_row, shap_row, feature_cols, customer_row['RiskTier']
)

print("Win-back email:")
print(winback_email)

Initializing LLM client: llama3
LLM client ready
Win-back email:
Hi there,

We've missed you! It's been a while since your last purchase, and we're excited to have you back. We noticed that you tend to spread out your shopping over time, and we want to make sure you don't miss out on any of the great deals and new arrivals.

As a valued customer, we'd like to offer you 15% off your next order. Just use code WELCOMEBACK at checkout to redeem your discount. We're looking forward to seeing what catches your eye!

The Store Team
